# Greenhouse RL Controller — PPO Training
**Phase 5 of the Greenhouse AI Controller project**

This notebook trains a PPO (Proximal Policy Optimization) agent to control
a greenhouse simulation. It then compares three agents:

| Agent | Method | Expected reward |
|-------|--------|-----------------|
| Random | No learning | ~100–200 |
| Q-Learning | Tabular (64 states) | ~200–300 |
| **PPO** | Neural network policy | **400–700+** |

**Why PPO beats Q-Learning:**
Q-Learning stores a table of 64 states. It fails for situations it rarely visits.
PPO uses a neural network that generalises across all situations it has seen.
It handles continuous state values directly — no binary encoding needed.

**Run all cells top to bottom. Expected total time: ~5 minutes.**

In [ ]:
# Install required libraries
# stable-baselines3: the PPO implementation
# gymnasium: the environment interface
!pip install stable-baselines3 gymnasium matplotlib numpy --quiet

## Step 1: Upload your environment file

Before running the next cell, upload `greenhouse_env.py`:

1. Look at the **left sidebar** in Colab
2. Click the **folder icon** (Files)
3. Click the **upload icon** (paper with arrow)
4. Select `greenhouse_env.py` from your `greenhouse-rl/code/` folder
5. Wait for the upload to finish, then run the next cell

In [ ]:
import os, sys

# Check the file was uploaded correctly
path = '/content/greenhouse_env.py'
if os.path.exists(path):
    print('✓ greenhouse_env.py found — ready to proceed')
else:
    print('✗ File not found. Please upload greenhouse_env.py first (see instructions above)')
    raise FileNotFoundError('greenhouse_env.py not found at /content/')

In [ ]:
# Import and run a quick sanity check on the environment
sys.path.insert(0, '/content')
from greenhouse_env import GreenhouseEnv, PARAMS, PARAM_NAMES, N_ACTIONS

env = GreenhouseEnv()
obs, info = env.reset(seed=42)

print('Environment loaded successfully')
print(f'  Observation space: {env.observation_space}  (7 normalised sensor readings)')
print(f'  Action space:      {env.action_space}  (6 discrete actuator controls)')
print(f'  Episode length:    144 steps = 24 simulated hours')
print()
print('Initial sensor state:')
for name, val in info['raw_state'].items():
    p = PARAMS.get(name, [0,1,0,1])
    if isinstance(p, list):
        in_opt = p[2] <= val <= p[3]
        print(f'  {name:<22}: {val:7.2f}  [{"OK " if in_opt else "OUT"}]')
    else:
        print(f'  {name:<22}: {val:7.2f}')

## Step 2: Train the PPO agent

**What PPO does differently from Q-Learning:**

- Uses a neural network (2 hidden layers, 64 units each) as the policy
- Takes the raw 7 continuous sensor values as input — no binary encoding
- The network outputs a probability distribution over 6 actions
- Trains on batches of 2048 steps collected from the environment
- The 'clip_range=0.2' prevents the policy from changing too much in one update,
  making training stable

**Training will take approximately 3–6 minutes on Colab free CPU.**
You will see progress printed every 2048 steps.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np

# This callback records episode rewards during training
# so we can plot the learning curve afterwards
class RewardLogger(BaseCallback):
    def __init__(self):
        super().__init__(verbose=0)
        self.episode_rewards = []
        self._current_ep_reward = 0.0

    def _on_step(self) -> bool:
        self._current_ep_reward += self.locals['rewards'][0]
        if self.locals['dones'][0]:
            self.episode_rewards.append(self._current_ep_reward)
            self._current_ep_reward = 0.0
        return True

print('RewardLogger callback defined')

In [ ]:
# Create a fresh environment for training
train_env = GreenhouseEnv()

# Configure PPO
# These are standard hyperparameters that work well for small custom environments
model = PPO(
    policy        = 'MlpPolicy',   # Multi-layer perceptron neural network
    env           = train_env,
    learning_rate = 3e-4,          # How fast the network weights update
    n_steps       = 2048,          # Steps collected before each update
    batch_size    = 64,            # Mini-batch size for gradient updates
    n_epochs      = 10,            # How many passes over collected data
    gamma         = 0.95,          # Discount factor (same as Q-learning)
    clip_range    = 0.2,           # PPO clipping: prevents policy changing too fast
    verbose       = 1,             # Print progress
    tensorboard_log = './ppo_tb/', # TensorBoard logs
)

print('PPO model created')
print(f'Policy network: {model.policy}')
print()
print('Starting training — 200,000 timesteps (~1,388 episodes)...')
print('This will take approximately 3–6 minutes.')
print()

reward_logger = RewardLogger()

model.learn(
    total_timesteps = 200_000,
    callback        = reward_logger,
    progress_bar    = True,
)

# Save the trained model
model.save('greenhouse_ppo_model')
print()
print(f'Training complete. {len(reward_logger.episode_rewards)} episodes completed.')
print(f'Model saved as: greenhouse_ppo_model.zip')

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

ppo_rewards = reward_logger.episode_rewards

def rolling_avg(data, window=20):
    return [np.mean(data[max(0,i-window+1):i+1]) for i in range(len(data))]

rolling = rolling_avg(ppo_rewards, window=20)

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_facecolor('#0d1f16')
fig.patch.set_facecolor('#0d1f16')

eps = range(1, len(ppo_rewards)+1)
ax.plot(eps, ppo_rewards, color='#52b788',
        linewidth=0.8, alpha=0.3, label='Episode reward')
ax.plot(eps, rolling, color='#b7e4c7', linewidth=2.5, label='20-episode average')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.4)

ax.set_xlabel('Episode', fontsize=11, color='#b7e4c7')
ax.set_ylabel('Total reward', fontsize=11, color='#b7e4c7')
ax.set_title('PPO Agent: Learning Curve', fontsize=13, fontweight='bold', color='#b7e4c7')
ax.tick_params(colors='#b7e4c7')
ax.spines[['bottom','left']].set_color('#2d6a4f')
ax.spines[['top','right']].set_visible(False)
ax.legend(fontsize=10, facecolor='#132318', labelcolor='#b7e4c7')
ax.set_facecolor('#0d1f16')

first20_avg = np.mean(ppo_rewards[:20])
last20_avg  = np.mean(ppo_rewards[-20:])
improvement = last20_avg - first20_avg

plt.tight_layout()
plt.savefig('ppo_learning_curve.png', dpi=150, facecolor='#0d1f16')
plt.show()

print(f'First 20 episodes avg:  {first20_avg:.1f}')
print(f'Last  20 episodes avg:  {last20_avg:.1f}')
print(f'Improvement:            {improvement:+.1f}')

In [ ]:
# ── Evaluate all three agents on 100 episodes each ──────────────────────────
# This is the comparison that makes your result academically meaningful.
# Same environment, same number of episodes, different agents.

def run_episodes(agent_fn, n_episodes=100, seed_start=0):
    """Run n_episodes and return list of total rewards."""
    eval_env = GreenhouseEnv()
    rewards = []
    for i in range(n_episodes):
        obs, info = eval_env.reset(seed=seed_start+i)
        ep_reward = 0
        for _ in range(144):
            action = agent_fn(obs)
            obs, reward, done, _, info = eval_env.step(action)
            ep_reward += reward
            if done:
                break
        rewards.append(ep_reward)
    return rewards

# 1. Random agent
print('Evaluating random agent (100 episodes)...')
eval_env_tmp = GreenhouseEnv()
random_rewards = run_episodes(
    lambda obs: eval_env_tmp.action_space.sample(), n_episodes=100
)

# 2. Q-learning agent (re-train for fair comparison)
print('Training Q-learning agent (200 episodes) then evaluating...')

N_ST_Q, N_AC_Q = 64, 6
qt = np.zeros((N_ST_Q, N_AC_Q))
eps_q = 1.0
rng_q = np.random.default_rng(0)

def q_state_key(raw_state):
    key = 0
    for i, name in enumerate(PARAM_NAMES[:6]):
        p = PARAMS[name]
        if p[2] <= raw_state[name] <= p[3]:
            key |= (1 << i)
    return key

q_train_env = GreenhouseEnv()
for ep in range(200):  # train Q-learning
    _, info = q_train_env.reset()
    sk = q_state_key(info['raw_state'])
    for _ in range(144):
        a = rng_q.integers(N_AC_Q) if rng_q.random()<eps_q else int(np.argmax(qt[sk]))
        obs, r, done, _, info = q_train_env.step(int(a))
        sk2 = q_state_key(info['raw_state'])
        qt[sk,a] += 0.1*(r + 0.95*np.max(qt[sk2]) - qt[sk,a])
        sk = sk2
        if done: break
    eps_q = max(0.05, eps_q - 0.012)

# Evaluate Q-learning
q_eval_env = GreenhouseEnv()
ql_rewards = []
for i in range(100):
    _, info = q_eval_env.reset(seed=i)
    ep_r = 0
    sk = q_state_key(info['raw_state'])
    for _ in range(144):
        a = int(np.argmax(qt[sk]))
        obs, r, done, _, info = q_eval_env.step(a)
        sk = q_state_key(info['raw_state'])
        ep_r += r
        if done: break
    ql_rewards.append(ep_r)

# 3. PPO agent
print('Evaluating PPO agent (100 episodes)...')
ppo_eval_rewards = run_episodes(
    lambda obs: int(model.predict(obs, deterministic=True)[0]), n_episodes=100
)

print()
print('=== EVALUATION RESULTS ===')
print(f'Random agent:  mean={np.mean(random_rewards):>8.1f}  std={np.std(random_rewards):.1f}')
print(f'Q-Learning:    mean={np.mean(ql_rewards):>8.1f}  std={np.std(ql_rewards):.1f}')
print(f'PPO (trained): mean={np.mean(ppo_eval_rewards):>8.1f}  std={np.std(ppo_eval_rewards):.1f}')
print()
ppo_vs_ql = np.mean(ppo_eval_rewards) - np.mean(ql_rewards)
ppo_vs_rn = np.mean(ppo_eval_rewards) - np.mean(random_rewards)
print(f'PPO improvement over Q-Learning: {ppo_vs_ql:+.1f}')
print(f'PPO improvement over Random:     {ppo_vs_rn:+.1f}')

In [ ]:
# Comparison bar chart — this is the key result figure for your paper

agents      = ['Random\nagent', 'Q-Learning\n(tabular)', 'PPO\n(neural net)']
means       = [np.mean(random_rewards), np.mean(ql_rewards), np.mean(ppo_eval_rewards)]
stds        = [np.std(random_rewards),  np.std(ql_rewards),  np.std(ppo_eval_rewards)]
colors      = ['#6b7280', '#e9c46a', '#52b788']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d1f16')

# Left: bar chart
ax1.set_facecolor('#0d1f16')
bars = ax1.bar(agents, means, color=colors, width=0.5,
               yerr=stds, capsize=6, error_kw={'color':'white','linewidth':1.2})
ax1.set_ylabel('Mean episode reward (100 episodes)', fontsize=10, color='#b7e4c7')
ax1.set_title('Agent comparison: mean reward', fontsize=12, fontweight='bold', color='#b7e4c7')
ax1.tick_params(colors='#b7e4c7')
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['bottom','left']].set_color('#2d6a4f')
for bar, mean, std in zip(bars, means, stds):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+std+8,
             f'{mean:.0f}', ha='center', va='bottom', fontsize=10,
             color='white', fontweight='bold')

# Right: reward distribution box plot
ax2.set_facecolor('#0d1f16')
bp = ax2.boxplot([random_rewards, ql_rewards, ppo_eval_rewards],
                 labels=agents, patch_artist=True,
                 medianprops={'color':'white','linewidth':2},
                 whiskerprops={'color':'#b7e4c7'},
                 capprops={'color':'#b7e4c7'},
                 flierprops={'marker':'o','markerfacecolor':'#b7e4c7','markersize':3})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_ylabel('Episode reward distribution', fontsize=10, color='#b7e4c7')
ax2.set_title('Reward distribution (100 episodes each)', fontsize=12, fontweight='bold', color='#b7e4c7')
ax2.tick_params(colors='#b7e4c7')
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['bottom','left']].set_color('#2d6a4f')

plt.tight_layout()
plt.savefig('agent_comparison.png', dpi=150, facecolor='#0d1f16')
plt.show()
print('Chart saved: agent_comparison.png')

In [ ]:
import csv
from datetime import datetime

# Export all results as CSV files
# These are the downloadable data files for your project

# 1. PPO training curve
with open('ppo_training_rewards.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['episode', 'total_reward', 'rolling_avg_20'])
    rolling20 = rolling_avg(ppo_rewards, 20)
    for i, (r, ra) in enumerate(zip(ppo_rewards, rolling20)):
        w.writerow([i+1, round(r,2), round(ra,2)])
print('Saved: ppo_training_rewards.csv')

# 2. Evaluation comparison
with open('agent_comparison_results.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['agent', 'episode', 'reward'])
    for i,r in enumerate(random_rewards): w.writerow(['random',    i+1, round(r,2)])
    for i,r in enumerate(ql_rewards):     w.writerow(['q_learning', i+1, round(r,2)])
    for i,r in enumerate(ppo_eval_rewards):w.writerow(['ppo',       i+1, round(r,2)])
print('Saved: agent_comparison_results.csv')

# 3. Summary table
with open('results_summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['agent','mean_reward','std_reward','min_reward','max_reward','n_episodes'])
    for name, data in [('random',random_rewards),('q_learning',ql_rewards),('ppo',ppo_eval_rewards)]:
        w.writerow([name, round(np.mean(data),2), round(np.std(data),2),
                    round(np.min(data),2), round(np.max(data),2), len(data)])
print('Saved: results_summary.csv')

print()
print('All files ready. To download from Colab:')
print('  Left sidebar → Files → right-click each file → Download')
print('  Save to your greenhouse-rl/results/ folder')

In [ ]:
# Model size check — important for Edge AI / ESP32 deployment feasibility
import os

model_size_kb = os.path.getsize('greenhouse_ppo_model.zip') / 1024

print('=== Model Information ===')
print(f'File:           greenhouse_ppo_model.zip')
print(f'Size:           {model_size_kb:.1f} KB')
print()
print('Policy architecture (MlpPolicy):')
print('  Input:         7 normalised sensor readings')
print('  Hidden layers: 2 x 64 units (tanh activation)')
print('  Output:        6 action probabilities')
print()

# Parameter count
total_params = sum(p.numel() for p in model.policy.parameters())
print(f'Total parameters: {total_params:,}')
print()

esp32_sram_kb = 520
tflite_est_kb = total_params * 4 / 1024  # float32 estimate
print('Edge AI feasibility (ESP32):')
print(f'  ESP32 SRAM:          {esp32_sram_kb} KB')
print(f'  TFLite model est:    {tflite_est_kb:.1f} KB (float32)')
print(f'  TFLite model est:    {tflite_est_kb/4:.1f} KB (int8 quantised)')
fits = tflite_est_kb/4 < 100
print(f'  Fits on ESP32:       {"Yes" if fits else "Tight — use int8 quantisation"}')

## Step 3: Extract weights for ESP32 deployment

The trained PPO policy is a small neural network: 5,062 parameters in the actor network.
This cell extracts those weights as C float arrays and saves them as `nn_weights.h`.

**What this does:**
- Extracts only the actor network (critic is used for training only, not needed on hardware)
- Writes all weight values as plain C float arrays
- Saves `nn_weights.h` to the Colab session
- Verifies the extraction by comparing a test inference against the SB3 model

**After running: download `nn_weights.h` from the Files panel (left sidebar). Place it next to `esp32_greenhouse_controller.ino`.**

In [ ]:
print('=== Phase 5 Complete ===')
print()
print('Files produced:')
print('  greenhouse_ppo_model.zip       — trained PPO model')
print('  ppo_learning_curve.png         — training curve')
print('  agent_comparison.png           — bar chart + box plot')
print('  ppo_training_rewards.csv       — episode-by-episode PPO rewards')
print('  agent_comparison_results.csv   — all three agents, 100 episodes each')
print('  results_summary.csv            — mean/std/min/max per agent')
print()
print('Download all files: Files panel (left sidebar) → right-click → Download')
print('Save to: greenhouse-rl/results/')
print()
print('Next: Phase 6 — Edge AI framing (ESP32 pseudocode, system diagram)')

In [ ]:
import numpy as np


def array_to_c(name, arr):
    flat   = arr.flatten()
    values = ', '.join(f'{v:.6f}f' for v in flat)
    shape  = arr.shape
    if len(shape) == 1:
        return f'const float {name}[{shape[0]}] = {{{values}}};'
    elif len(shape) == 2:
        return f'const float {name}[{shape[0]}][{shape[1]}] = {{{values}}};'


def verify_inference(W1, b1, W2, b2, W3, b3):
    obs = np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5], dtype=np.float32)
    h1  = np.tanh(W1 @ obs + b1)
    h2  = np.tanh(W2 @ h1  + b2)
    out = W3 @ h2 + b3
    python_action = int(np.argmax(out))
    import torch
    obs_tensor = torch.tensor(obs).unsqueeze(0)
    with torch.no_grad():
        sb3_action, _, _ = model.policy.forward(obs_tensor)
    sb3_action_int = int(sb3_action.squeeze().numpy())
    print(f'Python forward pass action: {python_action}')
    print(f'SB3 model action:           {sb3_action_int}')
    match = python_action == sb3_action_int
    print(f'Match: {chr(10004)+" Weights extracted correctly" if match else chr(10008)+" Mismatch"}')
    return match


print('Extracting actor network weights from trained PPO model...')
print()

policy = model.policy
actor_layers = list(policy.mlp_extractor.policy_net.children())

W1 = actor_layers[0].weight.detach().numpy().astype(np.float32)
b1 = actor_layers[0].bias.detach().numpy().astype(np.float32)
W2 = actor_layers[2].weight.detach().numpy().astype(np.float32)
b2 = actor_layers[2].bias.detach().numpy().astype(np.float32)
W3 = policy.action_net.weight.detach().numpy().astype(np.float32)
b3 = policy.action_net.bias.detach().numpy().astype(np.float32)

print('Layer shapes (actor network only):')
print(f'  W1 (input 7 -> hidden 64):  {W1.shape}   {W1.size} values')
print(f'  b1 (hidden 1 bias):         {b1.shape}    {b1.size} values')
print(f'  W2 (hidden 64 -> hidden 64):{W2.shape}  {W2.size} values')
print(f'  b2 (hidden 2 bias):         {b2.shape}    {b2.size} values')
print(f'  W3 (hidden 64 -> output 6): {W3.shape}    {W3.size} values')
print(f'  b3 (output bias):           {b3.shape}     {b3.size} values')
total = W1.size + b1.size + W2.size + b2.size + W3.size + b3.size
print(f'  Total actor parameters:     {total:,}')
print()
print('Note: full PPO model has 9,799 params (actor + critic).')
print('Only the actor (5,062 params) is needed for hardware inference.')
print()

print('Verifying weight extraction...')
verify_inference(W1, b1, W2, b2, W3, b3)
print()

header  = '// nn_weights.h\n'
header += '// Generated by greenhouse_rl_training.ipynb\n'
header += '// Actor network only: 5,062 parameters\n'
header += '// Forward pass: h1=tanh(W1*obs+b1), h2=tanh(W2*h1+b2), out=W3*h2+b3\n'
header += '// Action = argmax(out)\n'
header += '// Place alongside esp32_greenhouse_controller.ino\n\n'
header += '#pragma once\n\n'
header += array_to_c('W1', W1) + '\n\n'
header += array_to_c('b1', b1) + '\n\n'
header += array_to_c('W2', W2) + '\n\n'
header += array_to_c('b2', b2) + '\n\n'
header += array_to_c('W3', W3) + '\n\n'
header += array_to_c('b3', b3) + '\n'

with open('nn_weights.h', 'w') as f:
    f.write(header)

import os
size_kb = os.path.getsize('nn_weights.h') / 1024
print(f'Saved: nn_weights.h  ({size_kb:.1f} KB)')
print()
print('Download nn_weights.h from the Files panel (left sidebar).')
print('Place it in: code/esp32_greenhouse_controller/ alongside the .ino file.')
